   // AgriVision - Copyright (C) 2026 <Shahd Kazan>
   // Licensed under the GNU AGPL-3.0. See the LICENSE file.

In [ ]:
# Install MLflow in Kaggle Notebook

!pip install -q mlflow

# Verify installation
import mlflow
print("MLflow Version:", mlflow.__version__)

In [ ]:
# =============================================================
# Leaf Disease Instance Segmentation — YOLOv8
# FULL EXPERIMENTAL PIPELINE
# UPDATED BALANCED OVERSAMPLING VERSION
# =============================================================

# =============================================================
# 0. Setup
# =============================================================

import shutil
import os
import json
import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedShuffleSplit

import torch
import mlflow
import mlflow.pytorch

# =============================================================
# 1. Install dependencies
# =============================================================

!pip install -q ultralytics mlflow opencv-python-headless

from ultralytics import YOLO
from ultralytics import settings

settings.update({"mlflow": False})

os.environ["YOLO_VERBOSE"] = "False"

import warnings
warnings.filterwarnings("ignore")

# =============================================================
# MLflow setup
# =============================================================

mlflow.set_tracking_uri("file:/kaggle/working/mlruns")
mlflow.set_experiment("LeafDiseaseSegmentation")

# =============================================================
# 2. Paths
# =============================================================

DATA_ROOT = Path(
    "/kaggle/input/datasets/social888/plantdisease/leaf disease.yolov8"
)

WORKDIR = Path("/kaggle/working")

META_REGISTRY = WORKDIR / "experiment_registry.json"

SPLIT_DIR = WORKDIR / "splits"
PRED_DIR = WORKDIR / "predictions"

for p in [SPLIT_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# =============================================================
# 3. Reproducibility + memory optimization
# =============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("CUDA available:", torch.cuda.is_available())
print("Using device:", DEVICE)

def clear_memory(model=None):

    if model is not None:
        del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def hard_reset():

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# =============================================================
# 4. Load metadata
# =============================================================

meta = pd.read_csv(DATA_ROOT / "metadata.csv")

meta["strata"] = (
    meta["crop"].astype(str)
    + "_"
    + meta["label"].astype(str)
    + "_"
    + meta["environment"].astype(str)
)

# =============================================================
# 5. Fixed Stratified Split (NO LEAKAGE)
# =============================================================

split1 = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.1,
    random_state=SEED
)

train_val_idx, test_idx = next(
    split1.split(meta, meta["strata"])
)

train_val = meta.iloc[train_val_idx]
test = meta.iloc[test_idx]

split2 = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.111,
    random_state=SEED
)

train_idx, val_idx = next(
    split2.split(train_val, train_val["strata"])
)

train_df = train_val.iloc[train_idx].copy()
val_df = train_val.iloc[val_idx].copy()

# =============================================================
# 6. Dataset safety filter
# =============================================================

def valid_sample(img_name):

    img = DATA_ROOT / "train" / "images" / img_name

    lbl = (
        DATA_ROOT
        / "train"
        / "labels"
        / img_name.replace(".jpg", ".txt")
    )

    return img.exists() and lbl.exists()

train_df = train_df[
    train_df["image_name"].apply(valid_sample)
].copy()

val_df = val_df[
    val_df["image_name"].apply(valid_sample)
].copy()

test = test[
    test["image_name"].apply(valid_sample)
].copy()

# =============================================================
# 7. CREATE TWO TRAINING SETS
# =============================================================

# -------------------------------------------------------------
# ORIGINAL TRAIN SET
# -------------------------------------------------------------

train_df_original = train_df.copy()

# -------------------------------------------------------------
# UPDATED BALANCED TRAIN SET
# Better oversampling strategy
# -------------------------------------------------------------

train_df_balanced = train_df.copy()

# class + environment combination
train_df_balanced["balance_key"] = (
    train_df_balanced["label"].astype(str)
    + "_"
    + train_df_balanced["environment"].astype(str)
)

combo_counts = (
    train_df_balanced["balance_key"]
    .value_counts()
)

max_count = combo_counts.max()

balanced_parts = []

for combo_name, count in combo_counts.items():

    subset = train_df_balanced[
        train_df_balanced["balance_key"] == combo_name
    ]

    # stronger balancing but capped
    target_size = min(
        int(max_count * 0.75),
        int(count * 2.0)
    )

    sampled_subset = subset.sample(
        n=target_size,
        replace=True,
        random_state=SEED
    )

    balanced_parts.append(sampled_subset)

train_df_balanced = pd.concat(
    balanced_parts,
    ignore_index=True
)

train_df_balanced = train_df_balanced.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

print("\n=================================================")
print("ORIGINAL TRAIN SIZE")
print(len(train_df_original))

print("\nBALANCED TRAIN SIZE")
print(len(train_df_balanced))

print("\n=================================================")
print("BALANCED LABEL DISTRIBUTION")
print(
    train_df_balanced["label"]
    .value_counts()
)

print("\n=================================================")
print("BALANCED ENV DISTRIBUTION")
print(
    train_df_balanced["environment"]
    .value_counts()
)

# =============================================================
# 8. Dataset builder
# =============================================================

def copy_split(df, name):

    out = SPLIT_DIR / name

    if out.exists():
        shutil.rmtree(out)

    (out / "images").mkdir(parents=True, exist_ok=True)
    (out / "labels").mkdir(parents=True, exist_ok=True)

    copied = 0

    for idx, row in df.iterrows():

        img = row["image_name"]

        unique_img_name = f"{idx}_{img}"

        src_img = (
            DATA_ROOT
            / "train"
            / "images"
            / img
        )

        dst_img = (
            out
            / "images"
            / unique_img_name
        )

        shutil.copy(src_img, dst_img)

        lbl = img.replace(".jpg", ".txt")

        src_lbl = (
            DATA_ROOT
            / "train"
            / "labels"
            / lbl
        )

        dst_lbl = (
            out
            / "labels"
            / unique_img_name.replace(".jpg", ".txt")
        )

        if src_lbl.exists():
            shutil.copy(src_lbl, dst_lbl)

        copied += 1

    print(f"{name}: copied {copied} files")

# =============================================================
# COPY DATASETS
# =============================================================

copy_split(train_df_original, "train")

copy_split(train_df_balanced, "train_balanced")

copy_split(val_df, "val")

copy_split(test, "test")

# =============================================================
# 9. YOLO YAML CONFIGS
# =============================================================

CLASS_NAMES = """
names:
  0: maize fall armyworm
  1: maize leaf
  2: maize leaf blight
  3: potato late blight
  4: potato leaf
  5: potato pest damage
  6: tomato early blight
  7: tomato leaf
  8: tomato leaf miner
"""

# -------------------------------------------------------------
# NORMAL YAML
# -------------------------------------------------------------

yaml_path = WORKDIR / "data.yaml"

yaml_path.write_text(f"""
path: {SPLIT_DIR}

train: train/images
val: val/images
test: test/images

nc: 9

{CLASS_NAMES}
""")

# -------------------------------------------------------------
# BALANCED YAML
# -------------------------------------------------------------

balanced_yaml_path = WORKDIR / "data_balanced.yaml"

balanced_yaml_path.write_text(f"""
path: {SPLIT_DIR}

train: train_balanced/images
val: val/images
test: test/images

nc: 9

{CLASS_NAMES}
""")

# =============================================================
# 10. Split statistics logging
# =============================================================

def log_split_stats(df, name):

    return {

        "split": name,

        "size": len(df),

        "percent": (
            len(df) / len(meta) * 100
        ),

        "crop_distribution":
            df["crop"]
            .value_counts(normalize=True)
            .to_dict(),

        "label_distribution":
            df["label"]
            .value_counts(normalize=True)
            .to_dict(),

        "env_distribution":
            df["environment"]
            .value_counts(normalize=True)
            .to_dict(),
    }

split_stats = [

    log_split_stats(
        train_df_original,
        "train"
    ),

    log_split_stats(
        train_df_balanced,
        "train_balanced"
    ),

    log_split_stats(
        val_df,
        "val"
    ),

    log_split_stats(
        test,
        "test"
    )
]

pd.DataFrame(split_stats).to_json(
    WORKDIR / "split_statistics.json",
    orient="records",
    indent=2
)

# =============================================================
# 11. Experiment registry
# =============================================================

def save_registry(entry):

    data = []

    if META_REGISTRY.exists():
        data = json.loads(
            META_REGISTRY.read_text()
        )

    data.append(entry)

    META_REGISTRY.write_text(
        json.dumps(data, indent=2)
    )

# =============================================================
# 12. Domain evaluation
# =============================================================

def eval_by_domain(model, full_df, domain):

    subset = full_df[
        full_df["environment"] == domain
    ]

    temp_dir = SPLIT_DIR / f"eval_{domain}"

    if temp_dir.exists():
        shutil.rmtree(temp_dir)

    images_dir = temp_dir / "images"
    labels_dir = temp_dir / "labels"

    images_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    labels_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    for idx, row in subset.iterrows():

        img_name = row["image_name"]

        unique_name = f"{idx}_{img_name}"

        src_img = (
            DATA_ROOT
            / "train"
            / "images"
            / img_name
        )

        dst_img = (
            images_dir
            / unique_name
        )

        shutil.copy(src_img, dst_img)

        lbl_name = img_name.replace(".jpg", ".txt")

        src_lbl = (
            DATA_ROOT
            / "train"
            / "labels"
            / lbl_name
        )

        dst_lbl = (
            labels_dir
            / unique_name.replace(".jpg", ".txt")
        )

        if src_lbl.exists():
            shutil.copy(src_lbl, dst_lbl)

    temp_yaml = WORKDIR / f"{domain}_eval.yaml"

    temp_yaml.write_text(f"""
path: {temp_dir}

train: images
val: images

nc: 9

{CLASS_NAMES}
""")

    metrics = model.val(
        data=str(temp_yaml),
        split="val",
        batch=4,
        workers=2,
        verbose=False
    )

    return {

        "seg_map50":
            metrics.seg.map50,

        "seg_map5095":
            metrics.seg.map
    }

# =============================================================
# 13. TRAIN FUNCTION
# =============================================================

def train(
    model_name,
    img_size,
    exp_name,
    extra=None,
    resume=False,
    yaml_file=None,
    epochs=50
):

    model = YOLO(model_name)

    with mlflow.start_run(
        run_name=exp_name
    ):

        results = model.train(

            data=str(
                yaml_file or yaml_path
            ),

            imgsz=img_size,

            epochs=epochs,

            batch=4,

            workers=2,

            cache=False,

            patience=8,

            pretrained=True,

            resume=resume,

            device=DEVICE,

            verbose=False,

            freeze=(
                extra or {}
            ).get("freeze", None),

            **{
                k: v
                for k, v in (
                    extra or {}
                ).items()
                if k != "freeze"
            }
        )

        metrics = model.val(

            data=str(
                yaml_file or yaml_path
            ),

            split="test",

            batch=4,

            workers=2,

            verbose=True
        )

        field_metrics = eval_by_domain(
            model,
            test,
            "field"
        )

        lab_metrics = eval_by_domain(
            model,
            test,
            "lab"
        )

        print(f"\n=== {exp_name} RESULTS ===")

        print(
            f"Seg mAP50: "
            f"{metrics.seg.map50:.4f}"
        )

        print(
            f"Seg mAP50-95: "
            f"{metrics.seg.map:.4f}"
        )

        print(
            f"Field Seg mAP50: "
            f"{field_metrics['seg_map50']:.4f}"
        )

        print(
            f"Lab Seg mAP50: "
            f"{lab_metrics['seg_map50']:.4f}"
        )

        # =====================================================
        # LOGGING
        # =====================================================

        mlflow.log_param(
            "model",
            model_name
        )

        mlflow.log_param(
            "img_size",
            img_size
        )

        mlflow.log_param(
            "epochs",
            epochs
        )

        mlflow.log_metric(
            "seg_mAP50",
            metrics.seg.map50
        )

        mlflow.log_metric(
            "seg_mAP50_95",
            metrics.seg.map
        )

        mlflow.log_metric(
            "seg_precision",
            metrics.seg.mp
        )

        mlflow.log_metric(
            "seg_recall",
            metrics.seg.mr
        )

        mlflow.log_metric(
            "field_seg_mAP50",
            field_metrics["seg_map50"]
        )

        mlflow.log_metric(
            "lab_seg_mAP50",
            lab_metrics["seg_map50"]
        )

        save_registry({

            "exp": exp_name,

            "model": model_name,

            "imgsz": img_size,

            "seg_mAP50":
                float(metrics.seg.map50),

            "seg_mAP50_95":
                float(metrics.seg.map),

            "seg_precision":
                float(metrics.seg.mp),

            "seg_recall":
                float(metrics.seg.mr),

            "field_seg_mAP50":
                float(field_metrics["seg_map50"]),

            "lab_seg_mAP50":
                float(lab_metrics["seg_map50"]),
           
            "field_seg_mAP50_95": float(field_metrics["seg_map5095"]),
            "lab_seg_mAP50_95": float(lab_metrics["seg_map5095"])
        })

    clear_memory(model)

    hard_reset()

    return model, metrics

# =============================================================
# 14. EXPERIMENT 5 — BALANCED SAMPLING
# =============================================================

exp5 = dict(

    cls=1.2,

    box=7.5,

    dfl=1.5
)

m5, r5 = train(

    "yolov8n-seg.pt",

    640,

    "balanced_sampling_v2",

    exp5,

    yaml_file=balanced_yaml_path,

    epochs=50
)

# =============================================================
# 15. EXPERIMENT 6 — BETTER AUGMENTATION
# =============================================================

aug6 = dict(

    hsv_h=0.015,

    hsv_s=0.30,

    hsv_v=0.20,

    degrees=5,

    translate=0.03,

    scale=0.20,

    fliplr=0.5,

    flipud=0.0,

    mosaic=0.4,

    mixup=0.05,

    copy_paste=0.15,

    erasing=0.1,

    close_mosaic=20
)

m6, r6 = train(

    "yolov8n-seg.pt",

    640,

    "better_aug",

    aug6,

    epochs=50
)

# =============================================================
# 16. EXPERIMENT 7 — FREEZE BACKBONE
# =============================================================

exp7 = dict(
    freeze=10
)

m7, r7 = train(

    "yolov8n-seg.pt",

    640,

    "freeze_backbone",

    exp7,

    epochs=50
)

# =============================================================
# 17. FINAL SUMMARY EXPORT
# =============================================================

df = pd.DataFrame(
    json.loads(
        META_REGISTRY.read_text()
    )
)

summary_path = (
    WORKDIR
    / "all_experiments_summary.csv"
)

df.to_csv(
    summary_path,
    index=False
)

print("\nSaved experiment summary to:")
print(summary_path)